## Dataclass and DataLoader

### Dataset Class

**What it is:**

A Dataset is an abstract class in PyTorch used to represent your data (inputs + labels).

**Key methods/It defines :**

__init__() → which tells how does data should be loaded

__len__() → returns total number of samples

__getitem__(idx) → returns one sample (data + label) at index idx

**Why we use it:**

  - To organize and standardize data access
  - To support custom datasets (CSV, images, text, etc.)
  - Enables PyTorch to fetch data efficiently during training



*In Simple Terms ,The Dataset class is essentially a blueprint. When you create a custom Dataset, you decide how data is loaded and returned.*

------------------

### DataLoader

**What it is:**

DataLoader wraps around a Dataset and provides an iterable over batches of data.

**Key features:**

  - Batching (batch_size)
  - Shuffling (shuffle=True)
  - Parallel loading (num_workers)
  - Automatic iteration

**Why we use it:**

  - To load data in batches instead of one-by-one
  - To speed up training using parallelism
  - To shuffle data for better model generalization

  ----------------------------------

  *In Simple terms, The DataLoader wraps a Dataset and handles batching, shuffling, and parallel loading.*

**DataLoader Control Flow :**
  - At Start of the epoch , the DataLoader(if shuffles=True) shuffles indices(using a sampler).
  - It divides the indices into chunks of batch_size
  - for each index in the chunk, data samples are fetched from the Dataset object.
  - The samples are then collected and combined into a batch (using collate_fn)
  - The batch is returned to the main training loop.

**collate_fn:**

The collate_fn in PyTorch's Dataloader is a function that specifies how to combine a list of samples from a dataset into a single batch. By default, the DataLoader uses a simple batch collation mechansim, but collate_fn allows to customize how data should be processed and batched.

#### **DataLoader Important Parameters**

The DataLoader class in PyTorch comes with several parameters that allow you to customize how data is loaded, batched, and preprocessed. Some of the most commonly used and important parameters include:

1. **dataset (mandatory):** The Dataset from which the DataLoader will pull data. Must be a subclass of torch.utils.data.Dataset that implements _getitem_and _len_.

2. **batch_size:**
    - How many samples per batch to load.
    - Default is 1.
    - Larger batch sizes can speed up training on GPUs but require more memory.

3. **shuffle:**
    - If True, the DataLoader will shuffle the dataset indices each epoch.
    - Helpful to avoid the model becoming too dependent on the order of samples.

4. **num_workers:**
    - The number of worker processes used to load data in parallel.
    - Setting num_workers > 0 can speed up data loading by leveraging multiple CPU cores, especially if I/O or preprocessing is a bottleneck.

5. **pin_memory:**
    - If True, the DataLoader will copy tensors into pinned (page-locked) memory before returning them.
    - This can improve GPU transfer speed and thus overall training throughput, particularly on CUDA systems.

6. **drop_last:**
    - If True, the DataLoader will drop the last incomplete batch if the total number of samples is not divisible by the batch size.
    - Useful when exact batch sizes are required (for example, in some batch normalization scenarios).

7. **collate_fn:**
    - A callable that processes a list of samples into a batch (the default simply stacks tensors).
    - Custom collate_fn can handle variable-length sequences, perform custom batching logic, or handle complex data structures.

8. **sampler:**
    - sampler defines the strategy for drawing samples (e.g., for handling imbalanced classes, or custom sampling strategies).
    - batch_sampler works at the batch level, controlling how batches are formed.

Typically, we don't need to specify these if you are using batch_size and shuffle. However, they provide lower-level control if you have advanced requirements.

## A Simple Example

In [11]:
# Imports
import torch
import torch.nn as nn
import torch.optim as optim

##### Step 1: Create Dataset

In [8]:
# Features: [Age, Salary]
X = torch.tensor([
    [22.0, 20000.0],
    [25.0, 25000.0],
    [47.0, 60000.0],
    [52.0, 65000.0],
    [46.0, 70000.0],
    [56.0, 80000.0]
])

# Labels: 0 = No Buy, 1 = Buy
y = torch.tensor([
    [0.0],
    [0.0],
    [1.0],
    [1.0],
    [1.0],
    [1.0]
])

##### Step-2 : Custom Dataset with dataset class

In [12]:
from torch.utils.data import Dataset, DataLoader

class CustomerDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        print(f"Fetching index: {idx}")
        return self.features[idx], self.labels[idx]

#### Step 3: DataLoader

In [13]:
dataset = CustomerDataset(X, y)

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True   # triggers internal shuffling
)

#### Step 4: Defining Model

In [15]:
# -------------------------------
# Step 4: Model
# -------------------------------
class BinaryClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(2, 4), # 2 = num_features
            nn.Sigmoid(),
            nn.Linear(4, 1),
            nn.Sigmoid()   # needed for BCE
        )

    def forward(self, x):
        return self.network(x)


model = BinaryClassifier()

#### Step 5: Loss and Optimzer

In [18]:
## Important Parameters
learning_rate = 0.001
epochs = 5

In [19]:
# -------------------------------
# Step 5: Loss & Optimizer
# -------------------------------
loss_fn = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

#### Step 6: Training Pipeline

In [20]:
# -------------------------------
# Step 6: Training Loop
# -------------------------------

for epoch in range(epochs):
    print(f"\n--- Epoch {epoch+1} ---")

    for batch_idx, (x_batch, y_batch) in enumerate(loader):

        # Forward pass
        preds = model(x_batch)

        # Loss Calculation
        loss = loss_fn(preds, y_batch)

        # Clear Gradients
        optimizer.zero_grad()

        # Backward pass
        loss.backward()

        # Parameters Update
        optimizer.step()

        print(f"Batch {batch_idx+1} | Loss: {loss.item():.4f}")


--- Epoch 1 ---
Fetching index: 0
Fetching index: 5
Batch 1 | Loss: 0.6938
Fetching index: 1
Fetching index: 4
Batch 2 | Loss: 0.6938
Fetching index: 3
Fetching index: 2
Batch 3 | Loss: 0.7298

--- Epoch 2 ---
Fetching index: 2
Fetching index: 3
Batch 1 | Loss: 0.7287
Fetching index: 5
Fetching index: 4
Batch 2 | Loss: 0.7276
Fetching index: 1
Fetching index: 0
Batch 3 | Loss: 0.6608

--- Epoch 3 ---
Fetching index: 1
Fetching index: 3
Batch 1 | Loss: 0.6937
Fetching index: 5
Fetching index: 4
Batch 2 | Loss: 0.7275
Fetching index: 2
Fetching index: 0
Batch 3 | Loss: 0.6937

--- Epoch 4 ---
Fetching index: 3
Fetching index: 1
Batch 1 | Loss: 0.6937
Fetching index: 2
Fetching index: 0
Batch 2 | Loss: 0.6937
Fetching index: 4
Fetching index: 5
Batch 3 | Loss: 0.7264

--- Epoch 5 ---
Fetching index: 2
Fetching index: 4
Batch 1 | Loss: 0.7253
Fetching index: 0
Fetching index: 3
Batch 2 | Loss: 0.6936
Fetching index: 1
Fetching index: 5
Batch 3 | Loss: 0.6936


Here we are not defining or using any collate_fn , so dataloader will use a by Default collate_fn = torch.stack().


"If all samples have the same shape, PyTorch’s default collate_fn stacks them into batches automatically using torch.stack.”